# Label Encoding

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score

import warnings
warnings.filterwarnings('ignore')

# Generate dataset
np.random.seed(42)
n_samples = 1000


In [12]:
# Create sample dataset
np.random.seed(42)
data = pd.DataFrame({
    'city': ['NYC', 'LA', 'SF', 'NYC', 'LA', 'SF', 'NYC', 'Boston'] * 10,
    'education': ['High School', "Bachelor's", "Master's", 'PhD', "Bachelor's", "Master's", 'High School', "Bachelor's"] * 10,
    'satisfaction': ['Low', 'Medium', 'High', 'Medium', 'Low', 'High', 'Medium', 'Low'] * 10,
    'income': np.random.randint(30000, 120000, 80),
    'purchased': np.random.randint(0, 2, 80)
})


In [13]:
print("\n" + "="*60)
print(data.head(10))
print("="*60)
print(f"Dataset shape: {data.shape}")

# Split data
X = data.drop('purchased', axis=1)
y = data['purchased']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)


     city    education satisfaction  income  purchased
0     NYC  High School          Low   45795          0
1      LA   Bachelor's       Medium   30860          0
2      SF     Master's         High  106820          1
3     NYC          PhD       Medium   84886          1
4      LA   Bachelor's          Low   36265          0
5      SF     Master's         High  112386          1
6     NYC  High School       Medium   67194          1
7  Boston   Bachelor's          Low  117498          1
8     NYC  High School          Low   74131          0
9      LA   Bachelor's       Medium   90263          0
Dataset shape: (80, 5)


### Method 1: Using LabelEncoder (for single column or target variable)

In [14]:
print("\n" + "="*60)
print("Method 1: LabelEncoder for Target Variable")
print("="*60)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"\nOriginal target values: {y[:10].values}")
print(f"Encoded target values: {y_encoded[:10]}")
print(f"Classes: {label_encoder.classes_}")
print(f"Mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")



Method 1: LabelEncoder for Target Variable

Original target values: [0 0 1 1 0 1 1 1 0 0]
Encoded target values: [0 0 1 1 0 1 1 1 0 0]
Classes: [0 1]
Mapping: {np.int64(0): np.int64(0), np.int64(1): np.int64(1)}


In [15]:
# Inverse transform
y_decoded = label_encoder.inverse_transform(y_encoded[:10])
print(f"Decoded back: {y_decoded}")


Decoded back: [0 0 1 1 0 1 1 1 0 0]


### Method 2: LabelEncoder for Features (nominal - problematic with linear models)

In [16]:

print("\n" + "="*60)
print("Method 2: LabelEncoder for Nominal Feature (City)")
print("="*60)

city_encoder = LabelEncoder()
city_encoded = city_encoder.fit_transform(X_train['city'])

print(f"\nOriginal cities: {X_train['city'].values[:10]}")
print(f"Encoded cities: {city_encoded[:10]}")
print(f"Mapping: {dict(zip(city_encoder.classes_, city_encoder.transform(city_encoder.classes_)))}")
print("\n⚠️ Warning: This creates arbitrary ordering (Boston=0, LA=1, NYC=2, SF=3)")
print("This can mislead linear models into thinking SF is '3x more' than Boston!")



Method 2: LabelEncoder for Nominal Feature (City)

Original cities: ['LA' 'NYC' 'SF' 'Boston' 'SF' 'NYC' 'SF' 'NYC' 'NYC' 'Boston']
Encoded cities: [1 2 3 0 3 2 3 2 2 0]
Mapping: {'Boston': np.int64(0), 'LA': np.int64(1), 'NYC': np.int64(2), 'SF': np.int64(3)}

⚠️ Warning: This creates arbitrary ordering (Boston=0, LA=1, NYC=2, SF=3)
This can mislead linear models into thinking SF is '3x more' than Boston!


### Method 3: OrdinalEncoder for Ordinal Features (proper ordering)

In [17]:
print("\n" + "="*60)
print("Method 3: OrdinalEncoder for Ordinal Feature (Education)")
print("="*60)

# Define the correct order
education_order = [['High School', "Bachelor's", "Master's", 'PhD']]

ordinal_encoder = OrdinalEncoder(categories=education_order)
education_encoded = ordinal_encoder.fit_transform(X_train[['education']])

print(f"\nOriginal education: {X_train['education'].values[:10]}")
print(f"Encoded education: {education_encoded[:10].flatten()}")
print(f"Mapping (in correct order):")
for idx, cat in enumerate(education_order[0]):
    print(f"  {cat}: {idx}")


Method 3: OrdinalEncoder for Ordinal Feature (Education)

Original education: ["Bachelor's" 'High School' "Master's" "Bachelor's" "Master's"
 'High School' "Master's" 'High School' 'High School' "Bachelor's"]
Encoded education: [1. 0. 2. 1. 2. 0. 2. 0. 0. 1.]
Mapping (in correct order):
  High School: 0
  Bachelor's: 1
  Master's: 2
  PhD: 3


### Method 4: OrdinalEncoder with automatic alphabetical ordering

In [18]:
print("\n" + "="*60)
print("Method 4: OrdinalEncoder with Alphabetical Ordering")
print("="*60)

auto_encoder = OrdinalEncoder()
satisfaction_encoded = auto_encoder.fit_transform(X_train[['satisfaction']])

print(f"\nOriginal satisfaction: {X_train['satisfaction'].values[:10]}")
print(f"Encoded satisfaction: {satisfaction_encoded[:10].flatten()}")
print(f"Automatic mapping (alphabetical):")
for idx, cat in enumerate(auto_encoder.categories_[0]):
    print(f"  {cat}: {idx}")
print("\n⚠️ Warning: Alphabetical order gives High=0, Low=1, Medium=2")
print("This is WRONG for ordinal data! Always specify the correct order.")



Method 4: OrdinalEncoder with Alphabetical Ordering

Original satisfaction: ['Medium' 'Low' 'High' 'Low' 'High' 'Medium' 'High' 'Medium' 'Low' 'Low']
Encoded satisfaction: [2. 1. 0. 1. 0. 2. 0. 2. 1. 1.]
Automatic mapping (alphabetical):
  High: 0
  Low: 1
  Medium: 2

⚠️ Warning: Alphabetical order gives High=0, Low=1, Medium=2
This is WRONG for ordinal data! Always specify the correct order.


### Correct approach for satisfaction

In [20]:

satisfaction_order = [['Low', 'Medium', 'High']]
correct_encoder = OrdinalEncoder(categories=satisfaction_order)
satisfaction_correct = correct_encoder.fit_transform(X_train[['satisfaction']])

print(f"\nCorrected encoding with proper order:")
print(f"Encoded satisfaction: {satisfaction_correct[:10].flatten()}")
print(f"Correct mapping:")
for idx, cat in enumerate(satisfaction_order[0]):
    print(f"  {cat}: {idx}")


Corrected encoding with proper order:
Encoded satisfaction: [1. 0. 2. 0. 2. 1. 2. 1. 0. 0.]
Correct mapping:
  Low: 0
  Medium: 1
  High: 2


### Method 5: Comparing with Tree-based vs Linear Models

In [21]:

print("\n" + "="*60)
print("Method 5: Impact on Different Model Types")
print("="*60)

# Prepare data with label encoding for nominal feature (city)
preprocessor_label = ColumnTransformer(
    transformers=[
        ('city_label', OrdinalEncoder(), ['city']),
        ('education', OrdinalEncoder(categories=[education_order[0]]), ['education']),
        ('satisfaction', OrdinalEncoder(categories=[satisfaction_order[0]]), ['satisfaction']),
        ('num', 'passthrough', ['income'])
    ]
)

# Pipeline with Random Forest (handles label encoding well)
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor_label),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Pipeline with Logistic Regression (problematic for nominal categories)
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor_label),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Evaluate both
rf_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=5, scoring='accuracy')
lr_scores = cross_val_score(lr_pipeline, X_train, y_train, cv=5, scoring='accuracy')

print(f"\nRandom Forest with Label Encoding:")
print(f"  Mean CV Score: {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})")
print(f"  ✅ Tree models handle label-encoded nominal features well")

print(f"\nLogistic Regression with Label Encoding (nominal 'city'):")
print(f"  Mean CV Score: {lr_scores.mean():.4f} (+/- {lr_scores.std():.4f})")
print(f"  ⚠️ Linear models may struggle with label-encoded nominal features")



Method 5: Impact on Different Model Types

Random Forest with Label Encoding:
  Mean CV Score: 0.3833 (+/- 0.0850)
  ✅ Tree models handle label-encoded nominal features well

Logistic Regression with Label Encoding (nominal 'city'):
  Mean CV Score: 0.4333 (+/- 0.0624)
  ⚠️ Linear models may struggle with label-encoded nominal features


### Method 6: Handling Unseen Categories

In [22]:
print("\n" + "="*60)
print("Method 6: Handling Unseen Categories")
print("="*60)

# Fit encoder on limited categories
train_cities = pd.DataFrame({'city': ['NYC', 'LA', 'SF']})
test_cities = pd.DataFrame({'city': ['NYC', 'Chicago', 'SF', 'Miami']})

encoder_unknown = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
encoder_unknown.fit(train_cities)

try:
    encoded_test = encoder_unknown.transform(test_cities)
    print(f"\nTraining cities: {train_cities['city'].values}")
    print(f"Test cities: {test_cities['city'].values}")
    print(f"Encoded test: {encoded_test.flatten()}")
    print(f"\nMapping: NYC→{encoder_unknown.transform([['NYC']])[0][0]}, "
          f"LA→{encoder_unknown.transform([['LA']])[0][0]}, "
          f"SF→{encoder_unknown.transform([['SF']])[0][0]}")
    print(f"Unknown categories (Chicago, Miami) encoded as: -1")
except Exception as e:
    print(f"Error: {e}")


Method 6: Handling Unseen Categories

Training cities: ['NYC' 'LA' 'SF']
Test cities: ['NYC' 'Chicago' 'SF' 'Miami']
Encoded test: [ 1. -1.  2. -1.]

Mapping: NYC→1.0, LA→0.0, SF→2.0
Unknown categories (Chicago, Miami) encoded as: -1


### Method 7: pandas factorize (quick exploration)

In [23]:
print("\n" + "="*60)
print("Method 7: pandas factorize (Quick & Simple)")
print("="*60)

codes, uniques = pd.factorize(data['city'])
print(f"\nOriginal: {data['city'].values[:10]}")
print(f"Encoded: {codes[:10]}")
print(f"Unique categories: {uniques}")
print(f"\n⚠️ Note: factorize uses order-of-appearance, not alphabetical")


Method 7: pandas factorize (Quick & Simple)

Original: ['NYC' 'LA' 'SF' 'NYC' 'LA' 'SF' 'NYC' 'Boston' 'NYC' 'LA']
Encoded: [0 1 2 0 1 2 0 3 0 1]
Unique categories: Index(['NYC', 'LA', 'SF', 'Boston'], dtype='object')

⚠️ Note: factorize uses order-of-appearance, not alphabetical


# Advanced Example: Comparing Encoding Strategies

In [4]:
categories_low = ['A', 'B', 'C']
categories_high = [f'Cat_{i}' for i in range(50)]

data = pd.DataFrame({
    'nominal_low': np.random.choice(categories_low, n_samples),
    'nominal_high': np.random.choice(categories_high, n_samples),
    'ordinal': np.random.choice(['Low', 'Medium', 'High'], n_samples),
    'numeric': np.random.randn(n_samples)
})

# Create target with some patterns
data['target'] = (
    (data['nominal_low'] == 'A').astype(int) +
    (data['ordinal'] == 'High').astype(int) +
    np.random.randint(0, 2, n_samples)
) % 2

X = data.drop('target', axis=1)
y = data['target']

print("Comparing Label Encoding vs One-Hot Encoding")
print("="*60)
print(f"Dataset: {n_samples} samples")
print(f"Features: {X.columns.tolist()}")
print(f"Nominal (low card): {len(categories_low)} categories")
print(f"Nominal (high card): {len(categories_high)} categories")


Comparing Label Encoding vs One-Hot Encoding
Dataset: 1000 samples
Features: ['nominal_low', 'nominal_high', 'ordinal', 'numeric']
Nominal (low card): 3 categories
Nominal (high card): 50 categories


In [11]:

# Define preprocessing strategies
strategies = {
    'Label Encoding': ColumnTransformer([
        ('ordinal', OrdinalEncoder(categories=[['Low', 'Medium', 'High']]), ['ordinal']),
        ('nominal', OrdinalEncoder(), ['nominal_low', 'nominal_high']),
        ('num', 'passthrough', ['numeric'])
    ]),
    'One-Hot Encoding': ColumnTransformer([
        ('ordinal', OrdinalEncoder(categories=[['Low', 'Medium', 'High']]), ['ordinal']),
        ('nominal', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), 
         ['nominal_low', 'nominal_high']),
        ('num', 'passthrough', ['numeric'])
    ]),
    'Mixed (OHE for low, Label for high)': ColumnTransformer([
        ('ordinal', OrdinalEncoder(categories=[['Low', 'Medium', 'High']]), ['ordinal']),
        ('ohe_low', OneHotEncoder(drop='first', sparse_output=False), ['nominal_low']),
        ('label_high', OrdinalEncoder(), ['nominal_high']),
        ('num', 'passthrough', ['numeric'])
    ])
}

# Test with different algorithms
algorithms = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
}

In [12]:
results = []

for strategy_name, preprocessor in strategies.items():
    print(f"\n{strategy_name}:")
    print("-" * 40)
    
    for algo_name, algorithm in algorithms.items():
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', algorithm)
        ])
        
        scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
        mean_score = scores.mean()
        std_score = scores.std()
        
        results.append({
            'Strategy': strategy_name,
            'Algorithm': algo_name,
            'Mean_CV_Score': mean_score,
            'Std_CV_Score': std_score
        })
        
        print(f"  {algo_name:20s}: {mean_score:.4f} (+/- {std_score:.4f})")

# Create results DataFrame
results_df = pd.DataFrame(results)

print("\n" + "="*60)
print("Key Insights:")
print("="*60)
print("""
1. Tree-based models (RF, Decision Tree):
   - Perform well with BOTH label encoding and one-hot encoding
   - Label encoding is more memory-efficient for high cardinality
2. Logistic Regression:
   - Better with one-hot encoding for nominal features
   - Label encoding introduces false ordinal relationships
3. Mixed strategy:
   - Often optimal: OHE for low cardinality, Label for high cardinality
   - Balances interpretability and dimensionality
1. High cardinality features:
   - Label encoding avoids curse of dimensionality
   - But consider alternatives: target encoding, hashing, grouping
""")

# Display best strategy per algorithm
print("\nBest Strategy per Algorithm:")
print("-" * 40)
best_per_algo = results_df.loc[results_df.groupby('Algorithm')['Mean_CV_Score'].idxmax()]
for _, row in best_per_algo.iterrows():
    print(f"{row['Algorithm']:20s}: {row['Strategy']:30s} ({row['Mean_CV_Score']:.4f})")


Label Encoding:
----------------------------------------
  Random Forest       : 0.5100 (+/- 0.0197)
  Decision Tree       : 0.4910 (+/- 0.0369)
  Logistic Regression : 0.5180 (+/- 0.0240)

One-Hot Encoding:
----------------------------------------
  Random Forest       : 0.4950 (+/- 0.0164)
  Decision Tree       : 0.4910 (+/- 0.0422)
  Logistic Regression : 0.5150 (+/- 0.0210)

Mixed (OHE for low, Label for high):
----------------------------------------
  Random Forest       : 0.5190 (+/- 0.0292)
  Decision Tree       : 0.4970 (+/- 0.0256)
  Logistic Regression : 0.5110 (+/- 0.0191)

Key Insights:

1. Tree-based models (RF, Decision Tree):
   - Perform well with BOTH label encoding and one-hot encoding
   - Label encoding is more memory-efficient for high cardinality
2. Logistic Regression:
   - Better with one-hot encoding for nominal features
   - Label encoding introduces false ordinal relationships
3. Mixed strategy:
   - Often optimal: OHE for low cardinality, Label for high ca

# Real-world example: Customer satisfaction survey

In [15]:
data = pd.DataFrame({
    'satisfaction': ['Very Unsatisfied', 'Satisfied', 'Neutral', 'Very Satisfied',
                     'Unsatisfied', 'Satisfied', 'Very Satisfied', 'Neutral'] * 25,
    'education': ['High School', "Bachelor's", "Master's", 'PhD',
                  "Bachelor's", 'High School', "Master's", 'PhD'] * 25,
    'income_bracket': ['Low', 'Medium', 'High', 'High', 'Medium', 
                       'Low', 'High', 'Medium'] * 25,
    'age': [25, 35, 45, 55, 30, 40, 50, 60] * 25,
    'will_recommend': [0, 1, 0, 1, 0, 1, 1, 0] * 25
})

print("Custom Ordinal Encoding for Survey Data")
print("="*60)

# Define correct ordinal mappings
ordinal_mappings = {
    'satisfaction': ['Very Unsatisfied', 'Unsatisfied', 'Neutral', 'Satisfied', 'Very Satisfied'],
    'education': ['High School', "Bachelor's", "Master's", 'PhD'],
    'income_bracket': ['Low', 'Medium', 'High']
}

print("\nOrdinal Feature Mappings:")
for feature, order in ordinal_mappings.items():
    print(f"\n{feature}:")
    for idx, category in enumerate(order):
        print(f"  {idx}: {category}")

# Create preprocessor with correct ordinal encoding
preprocessor = ColumnTransformer([
    ('satisfaction', OrdinalEncoder(categories=[ordinal_mappings['satisfaction']]), 
     ['satisfaction']),
    ('education', OrdinalEncoder(categories=[ordinal_mappings['education']]), 
     ['education']),
    ('income', OrdinalEncoder(categories=[ordinal_mappings['income_bracket']]), 
     ['income_bracket']),
    ('age', 'passthrough', ['age'])
])

# Create and train model
X = data.drop('will_recommend', axis=1)
y = data['will_recommend']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=42))
])

pipeline.fit(X_train, y_train)

print(f"\nModel Training Accuracy: {pipeline.score(X_train, y_train):.4f}")
print(f"Model Testing Accuracy: {pipeline.score(X_test, y_test):.4f}")

# Show transformed data
X_train_transformed = preprocessor.fit_transform(X_train)
print(f"\nTransformed data shape: {X_train_transformed.shape}")
print(f"Original features: {X_train.shape[1]}")
print(f"Transformed features: {X_train_transformed.shape[1]}")
print("\n✅ Ordinal encoding preserves dimensionality (4 features → 4 features)")

Custom Ordinal Encoding for Survey Data

Ordinal Feature Mappings:

satisfaction:
  0: Very Unsatisfied
  1: Unsatisfied
  2: Neutral
  3: Satisfied
  4: Very Satisfied

education:
  0: High School
  1: Bachelor's
  2: Master's
  3: PhD

income_bracket:
  0: Low
  1: Medium
  2: High

Model Training Accuracy: 1.0000
Model Testing Accuracy: 1.0000

Transformed data shape: (160, 4)
Original features: 4
Transformed features: 4

✅ Ordinal encoding preserves dimensionality (4 features → 4 features)
